In [1]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
from preprocessing.tokenization import *
from pathlib import Path
from sklearn.metrics import f1_score

DATA_DIR = Path("data/final_dataset")
SPLITS_DIR = DATA_DIR / "splits"
TK_DIR = DATA_DIR / "tokenizers"

In [ ]:
def make_vectorizer(vocab:str)->TfidfVectorizer:
    with open(vocab, "r") as f:
        tcfg = TokenizerConfig.from_json(f.read())

    tokenizer = Tokenizer(tcfg, False)
    return TfidfVectorizer(
        tokenizer= lambda text: tokenizer.text_to_tokens(text),
        vocabulary=tcfg.tokens.tolist(),
        analyzer="word",
        max_features=10_000,
        token_pattern=None,
        lowercase=False,
        preprocessor=lambda x: x
    )

vectorizer = make_vectorizer(TK_DIR / "top10k.json")
relieable = ["reliable", "political", "clickbait", "bias"]



def extract_df(df : pd.DataFrame, is_train : bool = False):
    print("Categorizing type...")
    y = df["type"].isin(["reliable", "political", "clickbait", "bias"]).to_numpy(dtype=int)

    print("Vectorizing content tokens...")
    if is_train: X_content = vectorizer.fit_transform(df["content"])
    else: X_content = vectorizer.transform(df["content"])

    return X_content, y

In [3]:
train_df = pd.read_csv("data/final_dataset/splits/train.csv")
X_train, y_train = extract_df(train_df, is_train=True)

X_val, y_val = extract_df(pd.read_csv("data/final_dataset/splits/val.csv"))

Categorizing type...
Vectorizing content tokens...


c:\Users\nva\Desktop\GDS2026\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\nva\Desktop\GDS2026\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:1378: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


KeyboardInterrupt: 

In [24]:
vectorizer

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",<function mak...002AC3C6D7530>
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyz

In [26]:
tk = vectorizer.build_tokenizer()
vectorizer.fit_transform(tk("hello world"))

c:\Users\nva\Desktop\GDS2026\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\nva\Desktop\GDS2026\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:1378: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


AttributeError: 'int' object has no attribute 'lower'

In [5]:
from sklearn.linear_model import LogisticRegression

logre_model = LogisticRegression(max_iter=500, C=5, class_weight='balanced')
logre_model.fit(X_train, y_train)

print(f1_score(y_val, logre_model.predict(X_val)))

# 805 -> 823 -> 838 -> 847 ->rel 872

0.8720271207738322


In [6]:
mnb_model = MultinomialNB(alpha=1e-100)

mnb_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, mnb_model.predict(X_val))}")

f1 score: 0.8249508067689886


In [7]:
cnb_model = ComplementNB(alpha=1e-100)

cnb_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, cnb_model.predict(X_val))}")

f1 score: 0.8269176081280215


c:\Users\andre\KU\GDS\GDS2026\.venv\Lib\site-packages\sklearn\naive_bayes.py:1077: RuntimeWarning: divide by zero encountered in log
  logged = np.log(comp_count / comp_count.sum(axis=1, keepdims=True))


In [8]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(penalty='l1', C=1, class_weight='balanced')
svm_model.fit(X_train, y_train)

print(f"f1 score: {f1_score(y_val, svm_model.predict(X_val))}")

# 808 -> 829 -> 841 -> 847 | 852 ->rel 876

f1 score: 0.8756456306861383


In [14]:
import joblib

joblib.dump(svm_model, "data/models/linearSVC.joblib")
joblib.dump(vectorizer, "data/models/linearSVC_vectorizer.joblib")

['data/models/linearSVC_vectorizer.joblib']

In [10]:
# import json

# with open("data/models/linearSVC.json", 'w') as f:
#     json.dump({
#     'classes' : svm_model.classes_.tolist(),
#     'intercept' : svm_model.intercept_.tolist(),
#     'coef' : svm_model.coef_.tolist(),
# }, f, indent=4)

# vectorizer.get

In [11]:
# model = LinearSVC()
# model.coef_ = svm_model.coef_
# model.intercept_ = svm_model.intercept_
# model.classes_ = svm_model.classes_

# print(f"f1 score: {f1_score(y_val, model.predict(X_val))}")